### Load Transcript


In [1]:
from langchain_community.document_loaders import YoutubeLoader

In [3]:
url="https://www.youtube.com/watch?v=snCDdJdiioI&list=WL&index=14&t=2873s"
loader=YoutubeLoader.from_youtube_url(url)
docs=loader.load()
print(docs[0].page_content[:300])

[Music] you're listening to data framed a podcast by datacamp in this show you'll hear all the latest trends and insights in data science whether you're just getting started in your data career or you're a data leader looking to scale data-driven decisions in your organization join us for in-depth d


### Text Splitting

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [5]:
splitter=RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=100
)

In [6]:
chunks=splitter.split_documents(docs)
print(len(chunks))

66


In [8]:
chunks[1]

Document(metadata={'source': 'snCDdJdiioI'}, page_content="I think that a B test is a dreadfully dull name for such an important type of data experiment however my campaign to rename is data driven face-offs is yet to gain much Headway so we shall have to stick to the existing terminology sharing her extensive experience in data experimentation is Anjali mirror the senior director of product analytics data science experimentation and instrumentation at DocuSign she's worked in marketing analytics and customer analytics and product analytics so she's got great insights into how a b tests can be used throughout all areas of business let's see what she has to say hi Anjali thank you for joining me here just to begin with can you tell us a little bit about what DocuSign does sure and now first of all just thanks for having me here happy to be here on your podcast so DocuSign and now I think at this time is the household name it's a market leader in e-signature we pioneered the e-signature 

### Embedding and vector store 

In [13]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

In [12]:
embedding=HuggingFaceEmbeddings()
vector_store = FAISS.from_documents(chunks,embedding)

### Retriever

In [14]:
from langchain_community import retrievers

In [16]:
retriever=vector_store.as_retriever(search_type="similarity",search_kwargs={"k":4})

In [17]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x00000240649AC050>, search_kwargs={'k': 4})

In [18]:
# Example
query="What is AB testing"
result=retriever.invoke(query)
print(result)

[Document(id='6e35f7f1-9ca2-4c41-8ab0-079b76c00f17', metadata={'source': 'snCDdJdiioI'}, page_content="incremental returns for the business so that's one of the I was the best practices now try to go Downstream as much as possible I mean again it's a fine balance right because it also like you know then that makes your test duration longer and comes at an opportunity cost but now it really tells you the truly incremental gains that you draw from the test and not just really shifting some user Behavior at a very like now upper funnel level which did not translate to gains Downstream all right so just going slightly more broader for context are there any other types of experiment people ought to know about Beyond a b tests yes no absolutely so I think a b testing yes is the most common use term and people understand that I mean there's an extension of a b tests called ABN ABC ABCD test right what essentially that means is instead of testing two groups now controller cast or Champion or C

### The retriever is giving the top 4 document which has the answere related to the above question 

### Augmentation

In [23]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate

load_dotenv()


True

In [22]:
llm=ChatGroq(model="llama-3.1-8b-instant",temperature=0.2)

In [24]:
prompt=PromptTemplate(
    template="""
Answer the question ONLY using the provided context.
If the answer is not in the context, say "I don't know".

{context}


Question:
{question}
""",
input_variables=["context","question"]
)

In [37]:
question="How do we monitor AB testing"
retrieved_docs=retriever.invoke(question)

In [38]:
retrieved_docs

[Document(id='f2c5d2b2-06ec-47ae-833e-073eaa2a93ad', metadata={'source': 'snCDdJdiioI'}, page_content="run an experiment to either prove the hypothesis or disprove the hypothesis in my opinion also I would say this is one of the best ways to really you know determine any causal impact to any feature or factor that you're testing so yeah in a nutshell that's how I will describe experimentation to measure the impact of any feature or variable that you're changing in the business okay supporting or refuting hypotheses that's a really great summary of the idea of an experiment so a b tests are maybe the most popular type of data experiment at least in product analytics can you tell us a bit about what is an A B test yes I think it's the most common now testing used in the business so again really to simplify like a B test now the name came from like now you have two different groups so idle you split your audience into two different groups let's call them group a or group a again a more co

It is giving 4 document in which the meaning of ab testing is explained.We will concat the page content from the above document and create a final prompt and then we will send it to the LLM

In [43]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"run an experiment to either prove the hypothesis or disprove the hypothesis in my opinion also I would say this is one of the best ways to really you know determine any causal impact to any feature or factor that you're testing so yeah in a nutshell that's how I will describe experimentation to measure the impact of any feature or variable that you're changing in the business okay supporting or refuting hypotheses that's a really great summary of the idea of an experiment so a b tests are maybe the most popular type of data experiment at least in product analytics can you tell us a bit about what is an A B test yes I think it's the most common now testing used in the business so again really to simplify like a B test now the name came from like now you have two different groups so idle you split your audience into two different groups let's call them group a or group a again a more common term in the business is like now a control and a test or control and a variation group and how\n\

In [54]:
final_prompt= prompt.invoke({"context":context_text,"question":question})


In [55]:
# Let's see how our final prompt look
final_prompt

StringPromptValue(text='\nAnswer the question ONLY using the provided context.\nIf the answer is not in the context, say "I don\'t know".\n\nrun an experiment to either prove the hypothesis or disprove the hypothesis in my opinion also I would say this is one of the best ways to really you know determine any causal impact to any feature or factor that you\'re testing so yeah in a nutshell that\'s how I will describe experimentation to measure the impact of any feature or variable that you\'re changing in the business okay supporting or refuting hypotheses that\'s a really great summary of the idea of an experiment so a b tests are maybe the most popular type of data experiment at least in product analytics can you tell us a bit about what is an A B test yes I think it\'s the most common now testing used in the business so again really to simplify like a B test now the name came from like now you have two different groups so idle you split your audience into two different groups let\'s 

### Genration


In [49]:
answere=llm.invoke(final_prompt)
print(answere.content)

You can monitor AB testing by keeping a close eye on the test while it's running. This typically involves the testing team, as well as stakeholders, monitoring the test and its results. Some companies also have self-service tools that allow external folks to monitor the test without needing to write queries. 

In terms of the reporting piece, it's common to break it down into three different steps: 

1. Early read: This is an initial look at the results to see if there's a significant difference between the control and test groups.
2. Mid-level read: This is a more in-depth analysis of the results to see if the early read holds up and to identify any potential issues.
3. Final read: This is the final analysis of the results to determine the outcome of the test and to identify any key takeaways.


Now we will convert into chain / rag pipeline resume the video from 25:15